# Intro

So far we've learnt how to scrape the web, and how to make a request for information from an API. Some websites make APIs even easier. Check out [RapidAPI](https://rapidapi.com/) they take care of writing most of the code for you.

We will use the [AeroDataBox API](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/), which can retrieve all sorts of information about flights and airports. We will show you how to retrieve information about the airports, and then it's up to you to apply this, along with what you've already learnt this week, to **produce a function, which retrieves tomorrows flight information for the major airports in the cities you web scraped**.

In [1]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv
import datetime

load_dotenv()

api_key = os.environ["API_KEY_OPENAEROBOX"]

querystring = {"lat":"52.13","lon":"13.24","radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

headers = {
	"x-rapidapi-key": api_key,
	"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	"Content-Type": "application/json"
}

On the left hand side of the AeroDataBox API page, you'll see a list of options for information that you can retrieve:
> - Flight API
> - Flight Alert API
> - Airport API
> - Aircraft API
> - Industry API
> - Miscellaneous API
> - Healthcheck & Status API

1. We want to select `Airport API`

2. Then within Airport API we want to select `Search airports by location`

3. Now in the middle third you'll want to select `Params` and enter the `latitude` and `longitude` of any city to test... we chose Berlin: latitude 52.31 longitude 13.24. Next we changed the `radiusKM` to only 50km. And finally set `withFlightInfoOnly` to true, so it will only return airports which have flight data (scheduled or live) available.

4. On the right hand third of the screen you should see a block of code that looks pretty unfamiliar. This is because by default the code is probably set to *(Shell) Curl*. However, we have the power to change this to familiar python. Select the dropdown box at the top of the code and select `python > requests`.

Now you can copy the code to your notebook and it should look a little something like the cell below:

In [ ]:
import http.client

conn = http.client.HTTPSConnection("aerodatabox.p.rapidapi.com")

headers = {
    'x-rapidapi-key': api_key,
    'x-rapidapi-host': "aerodatabox.p.rapidapi.com"
}

conn.request("GET", "/airports/search/location?lat=52.31&lon=13.24&radiusKm=50&limit=10&withFlightInfoOnly=true", headers=headers)

res = conn.getresponse()
data = res.read()

In [9]:
print(res)#.decode("utf-8"))

In [13]:
response = requests.get("https://aerodatabox.p.rapidapi.com/airports/search/location?lat=52.31&lon=13.24&radiusKm=50&limit=10&withFlightInfoOnly=true", headers=headers)

In [2]:
#url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

querystring = {"lat":"52.13","lon":"13.24","radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

headers = {
	"x-rapidapi-key": api_key,
	"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	"Content-Type": "application/json"
}

#response = requests.get(url, headers=headers, params=querystring)

#print(response.json())

We can now turn this into a dataframe using `.json_normalize()`

In [17]:
pd.json_normalize(response.json()['items'])

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889


Let's now use this for the latitude and longitude of multiple cities

In [3]:
def get_airports(latitudes, longitudes):
    # API headers
    headers = {
        "x-rapidapi-key": api_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    # DataFrame to store results
    all_airports = []

    for lat, lon in zip(latitudes, longitudes):
    # Construct the URL with the latitude and longitude
        url = url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {"lat":lat,"lon":lon,"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            all_airports.append(airports)

    return pd.concat(all_airports, ignore_index=True)

In [19]:
# coordinates for Berlin, Paris, London
latitudes = [52.5200, 48.8567, 51.5072]
longitudes = [13.4050, 2.3522, -0.1275]

get_airports(latitudes, longitudes)

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
1,LFPB,LBG,Paris -Le Bourget,-Le Bourget,Paris,FR,Europe/Paris,48.96940,2.441390
2,LFPO,ORY,Paris Orly,Orly,Paris,FR,Europe/Paris,48.72530,2.359440
3,LFPG,CDG,Paris Charles de Gaulle,Charles de Gaulle,Paris,FR,Europe/Paris,49.01280,2.549999
4,EGLC,LCY,London City,City,London,GB,Europe/London,51.50530,0.055277
5,EGLL,LHR,London Heathrow,Heathrow,London,GB,Europe/London,51.47060,-0.461941
6,EGKR,KRH,Redhill Aerodrome,Aerodrome,Redhill,GB,Europe/London,51.21360,-0.138611
7,EGKK,LGW,London Gatwick,Gatwick,London,GB,Europe/London,51.14810,-0.190277
8,EGGW,LTN,London Luton,Luton,London,GB,Europe/London,51.87470,-0.368333
9,EGSS,STN,London Stansted,Stansted,London,GB,Europe/London,51.88500,0.234999


# Challenge

###### **Challenge:** Arrivals information
Using what you have been shown above, plus the skills you've learnt in the last couple of days:
1. In `AeroDataBox API` use the `Flight API` > `FIDS/Schedules: Airport departures and arrivals (by time range)` section
2. Fill out the parameters in the middle third and then copy the `python: requests` code from the right hand third
3. Explore the data you get back. What would be useful in your DataFrame and what can be excluded? Remember Gans wants to know about when people are arriving in the city
4. Make a DataFrame from the information you see as important
5. Condense everything you did above into a function that can take a list of ICAO codes as an input, and as an output gives you a DataFrame with the information for *tomorrows arrivals*

In [1]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv
import datetime
import pymysql

load_dotenv()

api_key = os.environ["API_KEY_OPENAEROBOX"]
connection_string = f'mysql+pymysql://{os.environ["MYSQL_USER"]}:{os.environ["MYSQL_ROOT_PASSWORD"]}@{os.environ["MYSQL_HOST"]}:{os.environ["MYSQL_PORT"]}/gans' #/{schema}'

In [2]:
def get_airports(latitudes, longitudes):
    '''Retrieves airports nearby (50km) of the positions.

    latittudes, longitudes: lists containing the respective coordinates. Both must be equally long.

    Returns: pd.DataFrame
    '''
    # API headers
    headers = {
        "x-rapidapi-key": api_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    # DataFrame to store results
    all_airports = []

    for lat, lon in zip(latitudes, longitudes):
    # Construct the URL with the latitude and longitude
        url = url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {
            "lat":lat,
            "lon":lon,
            "radiusKm":"50",
            "limit":"10",
            "withFlightInfoOnly":"true",
        }

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            all_airports.append(airports)

    return pd.concat(all_airports, ignore_index=True)

def fetch_airports():
    '''Loads the city table from MySQL and returns the airports DataFrame, including the corresponding city_id.'''
    cities = pd.read_sql(
        "SELECT * FROM city",
        con = connection_string,
    )
    airports = get_airports(*cities.loc[:, ["latitude", "longitude"]].T.values.tolist())
    airports["city_id"] = cities["city_id"]
    return airports

def get_url(iata):
    '''Returns tuple of 12h URL for a given iata.'''
    t0 = datetime.date.today() + datetime.timedelta(days=1)
    t1 = datetime.datetime.today().replace(hour=23, minute=59) + datetime.timedelta(days=0, hours=12)
    t2 = t1 + datetime.timedelta(days=0, hours=0, minutes=1)
    t3 = t1 + datetime.timedelta(days=0, hours=12)
    fmt = lambda x: x.strftime("%Y-%m-%dT%H%%3A%M")
    return (
        f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{iata}/{fmt(t0)}/{fmt(t1)}",
        f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{iata}/{fmt(t2)}/{fmt(t3)}",
    )

def get_flight(url):
    querystring = {
        "withLeg":"true",
        "direction":"Arrival",
        "withCancelled":"true",
        "withCodeshared":"true",
        "withCargo":"false",
        "withPrivate":"false",
        "withLocation":"false",
    }

    headers = {
    	"x-rapidapi-key": api_key,
    	"x-rapidapi-host": "aerodatabox.p.rapidapi.com"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    df = pd.json_normalize(response.json()["arrivals"])
    if "arrival.runwayTime.local" in df.columns:
        revised_times = df.loc[:, "arrival.revisedTime.local"].fillna(df.loc[:, "arrival.runwayTime.local"])
        df = df.drop(["arrival.runwayTime.local", "arrival.runwayTime.utc"], axis=1)
    else:
        revised_times = df.loc[:, "arrival.revisedTime.local"]
    df["arrival"] = (
        df.loc[:, "arrival.scheduledTime.local"]
            .where(df.loc[:, "arrival.revisedTime.local"].isnull(), revised_times)
            .str.split("+").apply(lambda x: x[0])
    )
    return df

def get_flights(airports):
    '''Returns flights from the airports DataFrame querying the respective API.'''
    flights = []
    for iata in airports.loc[:, "iata"]:
        url1, url2 = get_url(iata)
        flights += [pd.concat([
            get_flight(url1),
            get_flight(url2),
        ])]
        flights[-1]["at_iata"] = iata
        flights[-1]["city_id"] = airports.loc[airports.loc[:, "iata"] == iata, "city_id"].values[0]

    return pd.concat(flights, ignore_index=True)

In [3]:
def flights_to_arrivals(df):
    df = df.drop_duplicates(subset=["callSign", "arrival"])
    arr = (
        df.loc[:, [
            'at_iata',
            'city_id',
            'departure.airport.icao',
            'departure.airport.iata',
            'departure.airport.name',
            'departure.airport.countryCode',
            'departure.airport.location.lat',
            'departure.airport.location.lon',
            'arrival',
        ]]
        .rename({
            'departure.airport.icao': 'icao',
            'departure.airport.iata': 'iata',
            'departure.airport.name': 'name',
            'departure.airport.countryCode': 'country_code',
            'departure.airport.location.lat': 'lat',
            'departure.airport.location.lon': 'lon',
        }, axis=1)
        .drop_duplicates([
            'iata',
            'arrival'
        ])
    )
    return arr

In [4]:
def get_arrivals():
    airports = fetch_airports()
    flights = get_flights(airports)
    return flights_to_arrivals(flights)

In [5]:
def write_arrivals(arrivals):
    arrivals.to_sql(
        'arrival_tmp',
        schema="gans",
        if_exists='replace',
        con=connection_string,
        index=False,
    )
    conn = pymysql.connect(
        host=os.environ["MYSQL_HOST"],
        port=int(os.environ["MYSQL_PORT"]),
        user=os.environ["MYSQL_USER"],
        password=os.environ["MYSQL_ROOT_PASSWORD"],
        database="gans",
    )
    
    with conn.cursor() as c:
        c.callproc("update_arrival")
    
    conn.commit()
    conn.close()

In [6]:
write_arrivals(get_arrivals())

In [5]:
arrivals = get_arrivals()

In [6]:
airports = fetch_airports()
airports

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon,city_id
0,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889,1
1,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229,2
2,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100,3


In [19]:
airports.loc[airports.loc[:, "iata"] == "MUC", "city_id"].values[0]

np.int64(3)

In [21]:
flights = get_flights(airports)
flights

,number,status,codeshareStatus,isCargo,departure.airport.icao,departure.airport.iata,departure.airport.name,departure.airport.location.lat,departure.airport.location.lon,departure.airport.countryCode,...,departure.revisedTime.utc,departure.revisedTime.local,departure.gate,departure.checkInDesk,arrival,at_iata,city_id,departure.runwayTime.utc,departure.runwayTime.local,departure.runway
0,XQ 966,Expected,IsOperator,False,LTBJ,ADB,İzmir,38.292400,27.157000,tr,...,NaN,NaN,NaN,NaN,2026-10-24 06:05,BER,1,NaN,NaN,NaN
1,SR 1599,Expected,IsOperator,False,OLBA,BEY,Beirut,33.820900,35.488400,lb,...,NaN,NaN,NaN,NaN,2026-10-24 06:10,BER,1,NaN,NaN,NaN
2,XQ 1774,Expected,IsOperator,False,LTDB,COV,Adana,36.894085,35.064663,tr,...,NaN,NaN,NaN,NaN,2026-10-24 06:45,BER,1,NaN,NaN,NaN
3,TK 3350,Expected,IsOperator,False,LTAI,AYT,Antalya,36.898700,30.800500,tr,...,NaN,NaN,NaN,NaN,2026-10-24 06:50,BER,1,NaN,NaN,NaN
4,W4 3109,Expected,IsOperator,False,LROP,OTP,Bucharest,44.572200,26.102200,ro,...,NaN,NaN,NaN,NaN,2026-10-24 07:00,BER,1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048,DE 1585,Expected,IsOperator,False,GCTS,TFS,Tenerife Island,28.044500,-16.572500,es,...,2026-10-24 16:40Z,2026-10-24 17:40+01:00,NaN,NaN,2026-10-24 23:10,MUC,3,NaN,NaN,NaN
1049,X3 2197,Expected,IsOperator,False,GCTS,TFS,Tenerife Island,28.044500,-16.572500,es,...,2026-10-24 16:35Z,2026-10-24 17:35+01:00,NaN,NaN,2026-10-24 23:15,MUC,3,NaN,NaN,NaN
1050,LH 2481,Expected,IsOperator,False,EGLL,LHR,London,51.470600,-0.461941,gb,...,2026-10-24 19:25Z,2026-10-24 20:25+01:00,NaN,D,2026-10-24 23:20,MUC,3,NaN,NaN,NaN
1051,LH 1799,Expected,IsOperator,False,LEPA,PMI,Palma De Mallorca,39.551700,2.738810,es,...,2026-10-24 19:15Z,2026-10-24 21:15+02:00,NaN,NaN,2026-10-24 23:25,MUC,3,NaN,NaN,NaN


In [37]:
arrivals = flights_to_arrivals(flights)

In [38]:
arrivals

,at_iata,city_id,icao,iata,name,country_code,lat,lon,arrival
0,BER,1,LTBJ,ADB,İzmir,tr,38.292400,27.157000,2026-10-24 06:05
1,BER,1,OLBA,BEY,Beirut,lb,33.820900,35.488400,2026-10-24 06:10
2,BER,1,LTDB,COV,Adana,tr,36.894085,35.064663,2026-10-24 06:45
3,BER,1,LTAI,AYT,Antalya,tr,36.898700,30.800500,2026-10-24 06:50
4,BER,1,LROP,OTP,Bucharest,ro,44.572200,26.102200,2026-10-24 07:00
...,...,...,...,...,...,...,...,...,...
1010,MUC,3,EBBR,BRU,Brussels,be,50.901400,4.484439,2026-10-24 20:50
1026,MUC,3,LTFM,IST,Istanbul,tr,41.275276,28.751944,2026-10-24 21:30
1049,MUC,3,GCTS,TFS,Tenerife Island,es,28.044500,-16.572500,2026-10-24 23:15
1050,MUC,3,EGLL,LHR,London,gb,51.470600,-0.461941,2026-10-24 23:20


In [41]:
write_arrivals(arrivals)

```sql
CREATE TABLE arrival (
    city_id INT NOT NULL,
    at_iata CHAR(3),
    icao CHAR(4),
    iata CHAR(3) NOT NULL,
    name TEXT,
    country_code CHAR(2),
    lat DECIMAL(10,7),
    lon DECIMAL(10,7),
    arrival VARCHAR(20) NOT NULL,
    FOREIGN KEY (city_id)
        REFERENCES city (city_id),
    UNIQUE (at_iata, iata, arrival)
);
```